# Retrieval-Augmented Generation (RAG)

In [23]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [24]:
# We populated the RAG with the data from the data/calories.csv file in
# the rag_setup.ipynb notebook
chroma_client = chromadb.PersistentClient(path="../chroma")
pregnancy_nutrition_db = chroma_client.get_collection(name="nutrition_qna")

In [25]:
results = pregnancy_nutrition_db.query(query_texts=["pregnancy"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print(sorted(results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

[('answer', 'A pregant woman is expected to gain around 1 to 2 kilograms in total during the first trimester of her pregancy.'), ('answer_length', 112), ('has_question_mark', True), ('keywords', 'pregancy kilograms in should around pregant expected 1 gained of during a the first woman be much to total by weight 2 months pregnancy trimester how gain three her is'), ('question', 'How much weight should be gained by a pregant woman in total during the first three months of her pregnancy?'), ('question_length', 108), ('topic', 'nutrition_qa')]
Question: How much weight should be gained by a pregant woman in total during the first three months of her pregnancy?
        Answer: A pregant woman is expected to gain around 1 to 2 kilograms in total during the first trimester of her pregancy.

        This Q&A pair provides information about nutrition and health topics.


[('answer', 'Anaemia can lead to difficulties in pregnancy and childbirth. Mothers with anaemia may have babies born without 

In [26]:
@function_tool
def pregnancy_nutrition_rag_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database that retrieves
    pregnancy-related nutrition questions and answers.

    Args:
        query: User's question related to pregnancy nutrition.
        max_results: Maximum number of relevant Q&A pairs to return.

    Returns:
        A formatted string containing pregnancy nutrition guidance.
    """

    results = pregnancy_nutrition_db.query(
        query_texts=[query],
        n_results=max_results
    )

    if not results["documents"][0]:
        return f"No pregnancy nutrition information found for: {query}"

    formatted_results = []

    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]

        question = metadata.get("question", "Question not available")
        answer = metadata.get("answer", "Answer not available")
        category = metadata.get("category", "General").title()

        formatted_results.append(
            f"Category: {category}\n"
            f"Q: {question}\n"
            f"A: {answer}"
        )

    return "Pregnancy Nutrition Information:\n\n" + "\n\n".join(formatted_results)


Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [27]:
pregnancy_nutrition_rag_tool('bananas')

TypeError: 'FunctionTool' object is not callable

In [28]:
pregnancy_nutrition_agent = Agent(
    name="Pregnancy Nutrition Assistant",
    instructions="""
    You are a helpful and responsible pregnancy nutrition assistant.
    You answer questions related to:
    - Nutrition during pregnancy
    - Foods to eat or avoid
    - Vitamins and supplements
    - Trimester-wise dietary guidance

    Rules:
    - Give clear, concise, and reassuring answers.
    - Use the pregnancy_nutrition_rag_tool when information lookup is required.
    - Do NOT provide medical diagnosis or prescriptions.
    - Always keep advice general and nutrition-focused.
    """,
    tools=[pregnancy_nutrition_rag_tool]
)


In [29]:
with trace("Pregnancy Nutrition Assistant with RAG"):
    result = await Runner.run(
        pregnancy_nutrition_agent,
        "What fruits are recommended during pregnancy and what nutrients do they provide?",
    )
    print(result.final_output)


Here are commonly recommended fruits during pregnancy and the nutrients they provide:

- Oranges, mandarins, grapefruits, lemons, limes (citrus fruits)
  - Vitamin C (helps iron absorption), hydration
- Strawberries, blueberries, raspberries, blackberries (berries)
  - Vitamin C, fiber, antioxidants
- Kiwifruit
  - Vitamin C, potassium, fiber
- Bananas
  - Potassium, quick source of energy, magnesium
- Apples, pears, grapes
  - Fiber, vitamin C (varies by fruit), hydration
- Mango, melon, watermelon, pineapple (ripe fruit variety)
  - Vitamin A (from mango), hydration, fiber (melon/pineapple depending on type)
- Dried fruits (e.g., raisins, apricots) in moderation
  - Iron (in small amounts), fiber, calories

Tips:
- Aim for a colorful mix to cover vitamins, minerals, and fiber.
- Pair iron-rich foods (like beans, lentils, fortified cereals) with vitamin C sources (citrus, berries) to boost iron absorption.
- Choose whole fruits over juice to get fiber and fewer added sugars.
- Wash fr